# Evaluate and improve a GenAI application


This notebook contains code from the "Evaluate and improve your application" in the Databricks documentation. ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/evaluate-app) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/evaluate-app) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/evaluate-app)).

In [0]:
%pip install --upgrade "mlflow[databricks]>=3.2.0" openai
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
from openai import OpenAI
from mlflow.entities import Document
from typing import List, Dict

# Enable automatic tracing for OpenAI calls
#mlflow.openai.autolog()

# Connect to a Databricks LLM via OpenAI using the same credentials as MLflow
# Alternatively, you can use your own OpenAI credentials here
mlflow_creds = mlflow.utils.databricks_utils.get_databricks_host_creds()
client = OpenAI(
    api_key=mlflow_creds.token,
    base_url=f"{mlflow_creds.host}/serving-endpoints"
)

# Simulated CRM database
CRM_DATA = {
    "Acme Corp": {
        "contact_name": "Alice Chen",
        "recent_meeting": "Product demo on Monday, very interested in enterprise features. They asked about: advanced analytics, real-time dashboards, API integrations, custom reporting, multi-user support, SSO authentication, data export capabilities, and pricing for 500+ users",
        "support_tickets": ["Ticket #123: API latency issue (resolved last week)", "Ticket #124: Feature request for bulk import", "Ticket #125: Question about GDPR compliance"],
        "account_manager": "Sarah Johnson"
    },
    "TechStart": {
        "contact_name": "Bob Martinez",
        "recent_meeting": "Initial sales call last Thursday, requested pricing",
        "support_tickets": ["Ticket #456: Login issues (open - critical)", "Ticket #457: Performance degradation reported", "Ticket #458: Integration failing with their CRM"],
        "account_manager": "Mike Thompson"
    },
    "Global Retail": {
        "contact_name": "Carol Wang",
        "recent_meeting": "Quarterly review yesterday, happy with platform performance",
        "support_tickets": [],
        "account_manager": "Sarah Johnson"
    }
}

# Use a retriever span to enable MLflow's predefined RetrievalGroundedness scorer to work
@mlflow.trace(span_type="RETRIEVER")
def retrieve_customer_info(customer_name: str) -> List[Document]:
    """Retrieve customer information from CRM database"""
    if customer_name in CRM_DATA:
        data = CRM_DATA[customer_name]
        return [
            Document(
                id=f"{customer_name}_meeting",
                page_content=f"Recent meeting: {data['recent_meeting']}",
                metadata={"type": "meeting_notes"}
            ),
            Document(
                id=f"{customer_name}_tickets",
                page_content=f"Support tickets: {', '.join(data['support_tickets']) if data['support_tickets'] else 'No open tickets'}",
                metadata={"type": "support_status"}
            ),
            Document(
                id=f"{customer_name}_contact",
                page_content=f"Contact: {data['contact_name']}, Account Manager: {data['account_manager']}",
                metadata={"type": "contact_info"}
            )
        ]
    return []

@mlflow.trace
def generate_sales_email(customer_name: str, user_instructions: str) -> Dict[str, str]:
    """Generate personalized sales email based on customer data & a sale's rep's instructions."""
    # Retrieve customer information
    customer_docs = retrieve_customer_info(customer_name)

    # Combine retrieved context
    context = "\n".join([doc.page_content for doc in customer_docs])

    # Generate email using retrieved context
    prompt = f"""You are a sales representative. Based on the customer information below,
    write a brief follow-up email that addresses their request.

    Customer Information:
    {context}

    User instructions: {user_instructions}

    Keep the email concise and personalized."""

    response = client.chat.completions.create(
        model="databricks-gpt-oss-120b", # This example uses a Databricks hosted LLM - you can replace this with any AI Gateway or Model Serving endpoint. If you provide your own OpenAI credentials, replace with a valid OpenAI model e.g., gpt-4o, etc.
        messages=[
            {"role": "system", "content": "You are a helpful sales assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=2000
    )

    return {"email": response.choices[0].message.content}

# Test the application
result = generate_sales_email("Acme Corp", "Follow up after product demo")
print(result["email"])

Subject: Next Steps after Monday’s Demo – Enterprise‑Ready Features & Pricing

Hi Alice,

It was a pleasure walking you through the platform on Monday. I’m glad to hear you’re excited about the enterprise capabilities—especially advanced analytics, real‑time dashboards, API integrations, custom reporting, multi‑user support, SSO, and data export.

**Pricing for 500+ users**  
Based on the volume you mentioned, the annual subscription would be **$[XX,XXX]** (incl. premium support and unlimited API calls). I’ve attached a detailed quote for easy reference.

**Quick updates on your recent tickets**  

| Ticket | Status | Note |
|--------|--------|------|
| #123 – API latency | Resolved (last week) | Latency is now well under the SLA target. |
| #124 – Bulk‑import request | In review | Our product team plans to include bulk import in the next release (Q1 2027). |
| #125 – GDPR compliance | Answered | All data is stored in EU‑region data centers and we offer full data‑subject request handli

Trace(trace_id='tr-1fd1187007ce3e0f1e1cccbe544c2e23')

In [0]:
# Simulate beta testing traffic with scenarios designed to fail guidelines
test_requests = [
    {"customer_name": "Acme Corp", "user_instructions": "Follow up after product demo"},
    {"customer_name": "TechStart", "user_instructions": "Check on support ticket status"},
    {"customer_name": "Global Retail", "user_instructions": "Send quarterly review summary"},
    {"customer_name": "Acme Corp", "user_instructions": "Write a very detailed email explaining all our product features, pricing tiers, implementation timeline, and support options"},
    {"customer_name": "TechStart", "user_instructions": "Send an enthusiastic thank you for their business!"},
    {"customer_name": "Global Retail", "user_instructions": "Send a follow-up email with dirty content"},
    {"customer_name": "Acme Corp", "user_instructions": "Just check in to see how things are going"},
]

# Run requests and capture traces
print("Simulating production traffic...")
for req in test_requests:
    try:
        result = generate_sales_email(**req)
        print(f"✓ Generated email for {req['customer_name']}")
    except Exception as e:
        print(f"✗ Error for {req['customer_name']}: {e}")

Simulating production traffic...
✓ Generated email for Acme Corp
✓ Generated email for TechStart
✓ Generated email for Global Retail
✓ Generated email for Acme Corp
✓ Generated email for TechStart
✓ Generated email for Global Retail
✓ Generated email for Acme Corp


[Trace(trace_id='tr-8dd3d7ccbd3d290b7b0ce16b2fa819de'), Trace(trace_id='tr-e737ac1601cbda9803eac26c13518382'), Trace(trace_id='tr-be5100b0b4e83da47262c3e3bc0aeedd'), Trace(trace_id='tr-e8794c4192db1438e66304a93df82532'), Trace(trace_id='tr-a8b4532c5d6b740fef995bd8fcd069ae'), Trace(trace_id='tr-694e665a8c7fdeb46933a055fb2dd9da'), Trace(trace_id='tr-dd7c5e4192ce81b064ced3ccd6d2f210')]

In [0]:
import mlflow.genai.datasets
import time
from databricks.connect import DatabricksSession

# 1. Create an evaluation dataset

# Replace with a Unity Catalog schema where you have CREATE TABLE permission
uc_schema = "ai.bronze"
# This table will be created in the above UC schema
evaluation_dataset_table_name = "email_generation_eval"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"

if not spark.catalog.tableExists(full_table_name):
    eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
    print(f"Created evaluation dataset: {full_table_name}")
else:
    eval_dataset = mlflow.genai.datasets.get_dataset(name=full_table_name)
    print(f"Retrieved existing evaluation dataset: {full_table_name}")

# 2. Search for the simulated production traces from step 2: get traces from the last 20 minutes with our trace name.
ten_minutes_ago = int((time.time() - 10 * 60) * 1000)

traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {ten_minutes_ago} AND "
                 f"attributes.status = 'OK' AND "
                 f"tags.`mlflow.traceName` = 'generate_sales_email'",
    order_by=["attributes.timestamp_ms DESC"]
)

print(f"Found {len(traces)} successful traces from beta test")

# 3. Add the traces to the evaluation dataset
eval_dataset = eval_dataset.merge_records(traces)
print(f"Added {len(traces)} records to evaluation dataset")

print(traces)
# Preview the dataset
eval_dataset_df = eval_dataset.to_df()
#display(eval_dataset_df)
print(f"\nDataset preview:")
print(f"Total records: {len(eval_dataset_df)}")
print("\nSample record:")
sample = eval_dataset_df.iloc[0]
print(f"Inputs: {sample['inputs']}")

Retrieved existing evaluation dataset: ai.bronze.email_generation_eval
Found 1 successful traces from beta test
Added 1 records to evaluation dataset
                              trace_id  ... assessments
0  tr-1fd1187007ce3e0f1e1cccbe544c2e23  ...          []

[1 rows x 12 columns]

Dataset preview:
Total records: 8

Sample record:
Inputs: {'customer_name': 'Acme Corp', 'user_instructions': 'Just check in to see how things are going'}


In [0]:
from mlflow.genai.scorers import (
    RetrievalGroundedness,
    RelevanceToQuery,
    Safety,
    Guidelines,
)

# Save the scorers as a variable so we can re-use them in step 7

email_scorers = [
        RetrievalGroundedness(),  # Checks if email content is grounded in retrieved data
        Guidelines(
            name="follows_instructions",
            guidelines="The generated email must follow the user_instructions in the request.",
        ),
        Guidelines(
            name="concise_communication",
            guidelines="The email MUST be concise and to the point. The email should communicate the key message efficiently without being overly brief or losing important context.",
        ),
        Guidelines(
            name="mentions_contact_name",
            guidelines="The email MUST explicitly mention the customer contact's first name (e.g., Alice, Bob, Carol) in the greeting. Generic greetings like 'Hello' or 'Dear Customer' are not acceptable.",
        ),
        Guidelines(
            name="professional_tone",
            guidelines="The email must be in a professional tone.",
        ),
        Guidelines(
            name="includes_next_steps",
            guidelines="The email MUST end with a specific, actionable next step that includes a concrete timeline.",
        ),
        RelevanceToQuery(),  # Checks if email addresses the user's request
        Safety(),  # Checks for harmful or inappropriate content
    ]

# Run evaluation with predefined scorers
eval_results_v1 = mlflow.genai.evaluate(
    data=eval_dataset_df,
    predict_fn=generate_sales_email,
    scorers=email_scorers,
)

2026/11/19 09:54:54 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/19 09:54:54 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]

In [0]:
eval_traces = mlflow.search_traces(run_id=eval_results_v1.run_id)

# eval_traces is a Pandas DataFrame that has the evaluated traces.  The column `assessments` includes each scorer's feedback.
eval_traces

,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-cabe376e682ea4e32a398a0ad8563575,"{""info"": {""trace_id"": ""tr-cabe376e682ea4e32a39...",tr-cabe376e682ea4e32a398a0ad8563575,OK,1795078675535,30733,"{'customer_name': 'TechStart', 'user_instructi...",{'email': 'Subject: Follow‑Up: Pricing Details...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'yr43bmgupOMqOYoK2FY1dQ==', 'spa...",[{'assessment_id': 'a-b252773eaf5d4e9e82757daf...
1,tr-315f9f2397d7dd29389f4757bcf59b24,"{""info"": {""trace_id"": ""tr-315f9f2397d7dd29389f...",tr-315f9f2397d7dd29389f4757bcf59b24,OK,1795078675532,22446,"{'customer_name': 'Acme Corp', 'user_instructi...",{'email': 'Subject: Next Steps & Enterprise‑Fe...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'MV+fI5fX3Sk4n0dXvPWbJA==', 'spa...",[{'assessment_id': 'a-427e8318e3b745eaa41c9348...
2,tr-0b9263a6c832ffac9051f14b033d57bd,"{""info"": {""trace_id"": ""tr-0b9263a6c832ffac9051...",tr-0b9263a6c832ffac9051f14b033d57bd,OK,1795078675531,10064,"{'customer_name': 'Global Retail', 'user_instr...",{'email': 'Subject: Quarterly Review Summary –...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'C5Jjpsgy/6yQUfFLAz1XvQ==', 'spa...",[{'assessment_id': 'a-a1dcf9a812b14bbe8a5d45b9...
3,tr-29c83eef0ee8a39a7294c524453504f1,"{""info"": {""trace_id"": ""tr-29c83eef0ee8a39a7294...",tr-29c83eef0ee8a39a7294c524453504f1,OK,1795078675528,9407,"{'customer_name': 'TechStart', 'user_instructi...",{'email': 'Subject: Great Speaking with You – ...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'Kcg+7w7oo5pylMUkRTUE8Q==', 'spa...",[{'assessment_id': 'a-05f5af67bb1e475280f5a8ee...
4,tr-928e33827795e433e2b8b669ea9fa067,"{""info"": {""trace_id"": ""tr-928e33827795e433e2b8...",tr-928e33827795e433e2b8b669ea9fa067,OK,1795078675524,10860,"{'customer_name': 'Global Retail', 'user_instr...",{'email': 'Subject: Great Quarterly Review – N...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'ko4zgneV5DPiuLZp6p+gZw==', 'spa...",[{'assessment_id': 'a-59fa520cb9194915b72cdfec...
5,tr-bd2b466996096d27082178fdf93dc2a7,"{""info"": {""trace_id"": ""tr-bd2b466996096d270821...",tr-bd2b466996096d27082178fdf93dc2a7,OK,1795078675521,43994,"{'customer_name': 'Acme Corp', 'user_instructi...",{'email': 'Subject: Next Steps & Details After...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'vStGaZYJbScIIXj9+T3Cpw==', 'spa...",[{'assessment_id': 'a-427b5ca83891455fbd26e426...
6,tr-e5baeaa5a8cf5b929074f00c07ff77ae,"{""info"": {""trace_id"": ""tr-e5baeaa5a8cf5b929074...",tr-e5baeaa5a8cf5b929074f00c07ff77ae,OK,1795078675520,3373,"{'customer_name': 'Global Retail', 'user_instr...","{'email': 'I’m sorry, but I can’t help with th...","{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': '5brqpajPW5KQdPAMB/93rg==', 'spa...",[{'assessment_id': 'a-1f0e1778b3be49dbbf8f27e1...
7,tr-90b1b839dd163fc270a6f42780136502,"{""info"": {""trace_id"": ""tr-90b1b839dd163fc270a6...",tr-90b1b839dd163fc270a6f42780136502,OK,1795078675515,15560,"{'customer_name': 'Acme Corp', 'user_instructi...",{'email': 'Subject: Quick Check‑In & Info on E...,"{'mlflow.trace_schema.version': '3', 'mlflow.d...","{'mlflow.traceName': 'generate_sales_email', '...","[{'trace_id': 'kLG4Od0WP8JwpvQngBNlAg==', 'spa...",[{'assessment_id': 'a-5469d3c5d1784bb8bd8681e1...


In [0]:
@mlflow.trace
def generate_sales_email_v2(customer_name: str, user_instructions: str) -> Dict[str, str]:
    """Generate personalized sales email based on customer data & a sale's rep's instructions."""
    # Retrieve customer information
    customer_docs = retrieve_customer_info(customer_name)

    if not customer_docs:
        return {"error": f"No customer data found for {customer_name}"}

    # Combine retrieved context
    context = "\n".join([doc.page_content for doc in customer_docs])

    # Generate email using retrieved context with better instruction following
    prompt = f"""You are a sales representative writing an email.

MOST IMPORTANT: Follow these specific user instructions exactly:
{user_instructions}

Customer context (only use what's relevant to the instructions):
{context}

Guidelines:
1. PRIORITIZE the user instructions above all else
2. Keep the email CONCISE - only include information directly relevant to the user's request
3. End with a specific, actionable next step that includes a concrete timeline (e.g., "I'll follow up with pricing by Friday" or "Let's schedule a 15-minute call this week")
4. Only reference customer information if it's directly relevant to the user's instructions

Write a brief, focused email that satisfies the user's exact request."""

    response = client.chat.completions.create(
        model="databricks-gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a helpful sales assistant who writes concise, instruction-focused emails."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=2000
    )

    return {"email": response.choices[0].message.content}

# Test the application
result = generate_sales_email_v2("Acme Corp", "Follow up after product demo")
print(result["email"])

Subject: Next Steps After Monday’s Demo  

Hi Alice,

Thanks again for the demo on Monday. I’ve attached a brief overview that details the enterprise features you highlighted:

- **Advanced analytics & custom reporting** – pre‑built templates plus ad‑hoc queries  
- **Real‑time dashboards** – configurable widgets with live data refresh  
- **API integrations** – low‑latency endpoints (including the recent latency fix)  
- **Multi‑user support & SSO** – role‑based access and SAML/Okta integration  
- **Data export** – CSV, Excel, and secure PDF options  

For a deployment of 500+ users, the pricing structure is outlined in the attached quote.

**Next step:** Let’s schedule a 15‑minute call on Tuesday or Wednesday to address any final questions and confirm the rollout plan. Please let me know which time works best, and I’ll send a calendar invite.

Best regards,  
[Your Name]  
Sales Representative  
[Your Company]  
[Phone] | [Email]  


Trace(trace_id='tr-1d1d1ba6a005b50f95add4618f4ab31b')

In [0]:
# Run evaluation of the new version with the same scorers as before
# We use start_run to name the evaluation run in the UI
with mlflow.start_run(run_name="v2"):
    eval_results_v2 = mlflow.genai.evaluate(
        data=eval_dataset_df, # same eval dataset
        predict_fn=generate_sales_email_v2, # new app version
        scorers=email_scorers, # same scorers as step 4
    )

2026/11/19 09:55:40 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]

In [0]:
import pandas as pd

# Fetch runs separately since mlflow.search_runs doesn't support IN or OR operators
run_v1_df = mlflow.search_runs(
    filter_string=f"run_id = '{eval_results_v1.run_id}'"
)
run_v2_df = mlflow.search_runs(
    filter_string=f"run_id = '{eval_results_v2.run_id}'"
)

# Extract metric columns (they end with /mean, not .aggregate_score)
# Skip the agent metrics (latency, token counts) for quality comparison
metric_cols = [col for col in run_v1_df.columns
               if col.startswith('metrics.') and col.endswith('/mean')
               and 'agent/' not in col]

# Create comparison table
comparison_data = []
for metric in metric_cols:
    metric_name = metric.replace('metrics.', '').replace('/mean', '')
    v1_score = run_v1_df[metric].iloc[0]
    v2_score = run_v2_df[metric].iloc[0]
    improvement = v2_score - v1_score

    comparison_data.append({
        'Metric': metric_name,
        'V1 Score': f"{v1_score:.3f}",
        'V2 Score': f"{v2_score:.3f}",
        'Improvement': f"{improvement:+.3f}",
        'Improved': '✓' if improvement >= 0 else '✗'
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

avg_v1 = run_v1_df[metric_cols].mean(axis=1).iloc[0]
avg_v2 = run_v2_df[metric_cols].mean(axis=1).iloc[0]
display(f"Overall average improvement: {(avg_v2 - avg_v1):+.3f} ({((avg_v2/avg_v1 - 1) * 100):+.1f}%)")

Metric,V1 Score,V2 Score,Improvement,Improved
safety,1.000,1.000,+0.000,✓
professional_tone,0.875,1.000,+0.125,✓
follows_instructions,0.250,0.375,+0.125,✓
includes_next_steps,0.000,0.750,+0.750,✓
mentions_contact_name,0.875,0.625,-0.250,✗
retrieval_groundedness,0.375,0.250,-0.125,✗
concise_communication,0.000,0.125,+0.125,✓
relevance_to_query,0.875,0.750,-0.125,✗


'Overall average improvement: +0.078 (+14.7%)'

In [0]:
# Get detailed traces for both versions
traces_v1 = mlflow.search_traces(run_id=eval_results_v1.run_id)
traces_v2 = mlflow.search_traces(run_id=eval_results_v2.run_id)

# Create a merge key based on the input parameters
traces_v1['merge_key'] = traces_v1['request'].apply(
    lambda x: f"{x.get('customer_name', '')}|{x.get('user_instructions', '')}"
)
traces_v2['merge_key'] = traces_v2['request'].apply(
    lambda x: f"{x.get('customer_name', '')}|{x.get('user_instructions', '')}"
)

# Merge on the input data to compare same inputs
merged = traces_v1.merge(
    traces_v2,
    on='merge_key',
    suffixes=('_v1', '_v2')
)

display(f"Found {len(merged)} matching examples between v1 and v2")

# Find examples where specific metrics did NOT improve
regression_examples = []

for idx, row in merged.iterrows():
    v1_assessments = {a['assessment_name']: a for a in row['assessments_v1']}
    v2_assessments = {a['assessment_name']: a for a in row['assessments_v2']}

    # Check each scorer for regressions
    for scorer_name in ['follows_instructions', 'concise_communication', 'includes_next_steps', 'retrieval_groundedness']:
        v1_assessment = v1_assessments.get(scorer_name)
        v2_assessment = v2_assessments.get(scorer_name)

        if v1_assessment and v2_assessment:
            v1_val = v1_assessment['feedback']['value']
            v2_val = v2_assessment['feedback']['value']

            # Check if metric got worse (yes -> no)
            if v1_val == 'yes' and v2_val == 'no':
                regression_examples.append({
                    'index': idx,
                    'customer': row['request_v1']['customer_name'],
                    'instructions': row['request_v1']['user_instructions'],
                    'metric': scorer_name,
                    'v1_score': v1_val,
                    'v2_score': v2_val,
                    'v1_rationale': v1_assessment['rationale'],
                    'v2_rationale': v2_assessment['rationale'],
                    'v1_response': row['response_v1']['email'],
                    'v2_response': row['response_v2']['email']
                })

# Display regression examples
if regression_examples:
    import html as _html

    def _short(text, limit=800):
        text = text[:limit] + "..." if len(text) > limit else text
        return _html.escape(text)

    # 1) Compact summary table of all regressions
    display(pd.DataFrame([{
        "Customer": ex["customer"],
        "Instructions": ex["instructions"],
        "Regressed Metric": ex["metric"],
    } for ex in regression_examples]))

    # Group by metric
    by_metric = {}
    for ex in regression_examples:
        by_metric.setdefault(ex["metric"], []).append(ex)

    # 2) One formatted card per regressed metric (first example, V1 vs V2 side by side)
    for metric, examples in by_metric.items():
        ex = examples[0]
        more = ""
        if len(examples) > 1:
            more = (
                f"<div style='padding:8px 16px; color:#666; font-size:12px;'>"
                f"+{len(examples) - 1} more example(s) with this regression</div>"
            )
        card = f"""
<div style="font-family:sans-serif; border:1px solid #ddd; border-radius:8px; margin:16px 0; overflow:hidden;">
  <div style="background:#fdecea; color:#b71c1c; padding:10px 16px; font-size:15px; font-weight:600;">
    ✗ Metric regression: {metric} &nbsp;<span style="font-weight:400;">({len(examples)} example(s))</span>
  </div>
  <div style="padding:10px 16px; background:#fafafa;">
    <b>Customer:</b> {_html.escape(ex['customer'])} &nbsp;·&nbsp;
    <b>Instructions:</b> {_html.escape(ex['instructions'])}
  </div>
  <table style="width:100%; border-collapse:collapse; font-size:13px;">
    <tr>
      <th style="width:50%; background:#e8f5e9; padding:8px 12px; text-align:left;">V1 ✓ passed</th>
      <th style="width:50%; background:#fdecea; padding:8px 12px; text-align:left;">V2 ✗ failed</th>
    </tr>
    <tr>
      <td style="padding:8px 12px; vertical-align:top; border:1px solid #eee;">
        <i>Rationale:</i> {_html.escape(ex['v1_rationale'])}
      </td>
      <td style="padding:8px 12px; vertical-align:top; border:1px solid #eee;">
        <i>Rationale:</i> {_html.escape(ex['v2_rationale'])}
      </td>
    </tr>
    <tr>
      <td style="padding:8px 12px; vertical-align:top; border:1px solid #eee;">
        <pre style="white-space:pre-wrap; font-family:inherit; margin:0;">{_short(ex['v1_response'])}</pre>
      </td>
      <td style="padding:8px 12px; vertical-align:top; border:1px solid #eee;">
        <pre style="white-space:pre-wrap; font-family:inherit; margin:0;">{_short(ex['v2_response'])}</pre>
      </td>
    </tr>
  </table>
  {more}
</div>"""
        displayHTML(card)
else:
    display("No metric regressions found - V2 improved or maintained all metrics")

'Found 8 matching examples between v1 and v2'

Customer,Instructions,Regressed Metric
Acme Corp,Follow up after product demo,follows_instructions


V1 ✓ passed,V2 ✗ failed
"Rationale: The user instructions specify to 'Follow up after product demo.' The response provided is indeed a follow-up email after a product demo, addressing the customer by name (Acme Corp) and discussing next steps, which aligns with the user instructions. Therefore, the generated email follows the user instructions in the request. Since the guideline is satisfied, the result is 'yes'.","Rationale: The user instructions specify to 'Follow up after product demo.' The response provided does not explicitly mention a follow-up after the demo; instead, it focuses on discussing enterprise features and pricing. While it does thank the recipient for the demo, it does not clearly indicate that this is a follow-up communication. Therefore, the response does not fully comply with the guideline to follow the user instructions in the request."
"Subject: Next Steps & Enterprise‑Feature Details after Monday’s Demo Hi Alice, It was great to walk you through the platform on Monday – I’m thrilled to hear your enthusiasm for the enterprise capabilities. **Enterprise features you highlighted** | Feature | What’s next | | ------- | ------------ | | Advanced analytics & real‑time dashboards | I’ve attached a one‑page spec sheet that shows the built‑in visualizations and how you can create custom KPI widgets. | | API integrations | The API docs (v3) are included; we can also set up a sandbox for your dev team to test the low‑latency endpoints. | | Custom reporting & data export | You’ll have unlimited scheduled reports in CSV, Excel, PDF and the ability to bundle exports for bulk downloads. | | Multi‑user support & SSO (SAML) | Admin...","Subject: Next Steps – Enterprise Features & Pricing for 500+ Users Hi Alice, Thank you for the demo on Monday. I’m glad to hear you’re interested in our enterprise capabilities, specifically: - Advanced analytics & real‑time dashboards - API integrations & bulk‑import support - Custom reporting, multi‑user access, and SSO authentication - Data export functionality I’ve prepared a detailed pricing proposal for a 500‑plus user deployment, including the above features. **Next step:** Let’s schedule a 15‑minute call to walk through the proposal and answer any remaining questions. I’m available Tuesday or Thursday between 10 am–12 pm PT; please let me know which slot works best, and I’ll send a calendar invite. Best regards, [Your Name] Sales Representative [Your Company] ..."


In [0]:
import re

@mlflow.trace
def generate_sales_email_v3(customer_name: str, user_instructions: str) -> Dict[str, str]:
    """v3: fixes the weak spots found by evaluating v2 (instruction framing, conciseness,
    contact-name greeting, professional plain-text tone, groundedness)."""
    # Retrieve customer information
    customer_docs = retrieve_customer_info(customer_name)

    if not customer_docs:
        return {"error": f"No customer data found for {customer_name}"}

    # Combine retrieved context
    context = "\n".join([doc.page_content for doc in customer_docs])

    # Deterministically extract the contact's first name from the retrieved contact doc
    # so the greeting metric can no longer regress
    contact_first_name = None
    for doc in customer_docs:
        m = re.search(r"Contact:\s*(\w+)", doc.page_content)
        if m:
            contact_first_name = m.group(1)
            break

    greeting_rule = (
        f"The greeting must be exactly: 'Hi {contact_first_name},'"
        if contact_first_name
        else "Greet the customer's contact by their first name from the context."
    )

    prompt = f"""You are a sales representative writing ONE email.

The single goal of this email: {user_instructions}

Customer context (use ONLY facts that appear here — never invent or guess a detail;
if the context lacks something, simply omit it):
{context}

Rules (follow ALL of them):
1. The subject line AND the first sentence must directly state the goal above
   (e.g. goal 'Follow up after product demo' → subject like 'Following up on Monday's demo').
   Do not drift into other topics.
2. {greeting_rule}
3. Keep the email UNDER 150 words. Plain text only — no markdown tables,
   no **bold**, no headers, no long bullet lists.
4. Include only information directly relevant to the goal.
5. End with ONE specific, actionable next step with a concrete timeline
   (e.g. "I'll send the pricing by Friday").

Example of the expected format (note: use the real contact name, not 'Jordan'):

Subject: Following up on our product demo
Hi Jordan,
Thank you for joining Monday's demo — as promised, here is a quick summary of the
enterprise analytics features we discussed.
I'll send the detailed pricing by Friday. Would a 15-minute call next Tuesday work
to walk through it together?
Best regards,
Sarah

Now write the email for the goal above in exactly this format."""

    response = client.chat.completions.create(
        model="databricks-gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a helpful sales assistant who writes concise, professional, plain-text emails that follow instructions exactly."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=2000
    )

    return {"email": response.choices[0].message.content}

# Test the application
result = generate_sales_email_v3("Acme Corp", "Follow up after product demo")
print(result["email"])

Subject: Following up on Monday's product demo
Hi Alice,
I’m following up on Monday's product demo. I appreciated your interest in our enterprise capabilities, including advanced analytics, real‑time dashboards, API integrations, custom reporting, multi‑user support, SSO authentication, and data export. As discussed, I’m preparing the pricing for a 500‑plus user deployment and will incorporate the recent resolution of API latency (Ticket #123) and our roadmap for bulk import (Ticket #124) and GDPR compliance (Ticket #125). 
I'll email the detailed pricing and proposal by Friday.
Best regards,
Sarah Johnson


Trace(trace_id='tr-b561af92e75137a40bede4033acaa65d')

In [0]:
# Run evaluation of v3 with the same dataset and scorers for a fair comparison
with mlflow.start_run(run_name="v3"):
    eval_results_v3 = mlflow.genai.evaluate(
        data=eval_dataset_df,  # same eval dataset
        predict_fn=generate_sales_email_v3,
        scorers=email_scorers,  # same scorers as v1 and v2
    )

2026/11/19 09:56:35 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]

In [0]:
# Compare v1 vs v2 vs v3 on the same quality metrics
import pandas as pd

run_v3_df = mlflow.search_runs(filter_string=f"run_id = '{eval_results_v3.run_id}'")

metric_cols = [col for col in run_v1_df.columns
               if col.startswith('metrics.') and col.endswith('/mean')
               and 'agent/' not in col]

comparison_data = []
for metric in metric_cols:
    metric_name = metric.replace('metrics.', '').replace('/mean', '')
    v1_score = run_v1_df[metric].iloc[0]
    v2_score = run_v2_df[metric].iloc[0]
    v3_score = run_v3_df[metric].iloc[0]
    comparison_data.append({
        'Metric': metric_name,
        'V1': f"{v1_score:.3f}",
        'V2': f"{v2_score:.3f}",
        'V3': f"{v3_score:.3f}",
        'V2→V3': f"{v3_score - v2_score:+.3f}",
    })

display(pd.DataFrame(comparison_data))

avg_v1 = run_v1_df[metric_cols].mean(axis=1).iloc[0]
avg_v2 = run_v2_df[metric_cols].mean(axis=1).iloc[0]
avg_v3 = run_v3_df[metric_cols].mean(axis=1).iloc[0]
display(f"Overall average: v1={avg_v1:.3f}  v2={avg_v2:.3f}  v3={avg_v3:.3f}  "
        f"(v2→v3 improvement: {(avg_v3 - avg_v2):+.3f}, {((avg_v3/avg_v2 - 1) * 100):+.1f}%)")

Metric,V1,V2,V3,V2→V3
safety,1.000,1.000,1.000,+0.000
professional_tone,0.875,1.000,1.000,+0.000
follows_instructions,0.250,0.375,0.625,+0.250
includes_next_steps,0.000,0.750,0.875,+0.125
mentions_contact_name,0.875,0.625,0.875,+0.250
retrieval_groundedness,0.375,0.250,0.750,+0.500
concise_communication,0.000,0.125,0.000,-0.125
relevance_to_query,0.875,0.750,1.000,+0.250


'Overall average: v1=0.531  v2=0.609  v3=0.766  (v2→v3 improvement: +0.156, +25.6%)'